# Машинное обучение
# Задание 1. Ridge-регрессия

Дата выдачи: 07.09.2026

### О задании

Вы научитесь:
 * подготавливать данные для обучения линейных моделей;
 * обучать Ridge-регрессию при помощи модуля scikit-learn;
 * вычислять ошибки на валидационной и тренировочной выборках;
 * проверять модель на переобучение;
 * оценивать качество модели

### Теоретическая база

#### 1. Линейная регрессия

Линейная регрессия моделирует зависимость числового целевого признака $y$ от признаков объекта $x_1, x_2, \ldots, x_p$:

$$
\hat{y}_i = w_0 + w_1x_{i1} + w_2x_{i2} + \ldots + w_px_{ip}.
$$

Здесь $\hat{y}_i$ — предсказанная цена квартиры, $w_0$ — свободный член, а $w_j$ — вес признака $x_j$. В этой работе используются признаки `rooms`, `total area`, `floor` и `complex rating`, а целевая переменная `price`.

В матричной форме модель записывается так:

$$
\hat{\mathbf{y}} = X\mathbf{w},
$$

где к матрице $X$ обычно добавляют столбец единиц для свободного члена. Обучение состоит в подборе весов, при которых предсказания максимально близки к настоящим значениям.

#### 2. Функция ошибки

Для оценки близости предсказаний к ответам используются следующие метрики:

* **SSE (Sum of Squared Errors)** — сумма квадратов ошибок:
  $$\mathrm{SSE} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2.$$
* **MSE (Mean Squared Error)** — средняя квадратичная ошибка:
  $$\mathrm{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2 = \frac{\mathrm{SSE}}{n}.$$
* **RMSE (Root Mean Squared Error)** — корень из средней квадратичной ошибки:
  $$\mathrm{RMSE} = \sqrt{\mathrm{MSE}}.$$

Чем меньше эти значения, тем ближе предсказания к реальным ценам. RMSE измеряется в тех же единицах, что и `price`, поэтому его удобнее всего интерпретировать как типичный размер ошибки прогноза. Абсолютные пороги качества зависят от масштаба цен; в этой работе важны также сравнение train и validation и изменение ошибок по итерациям.

#### 3. Ridge-регрессия и регуляризация

Обычная линейная регрессия минимизирует SSE. Ridge-регрессия добавляет штраф за слишком большие веса:

$$
J(\mathbf{w}) = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha\sum_{j=1}^{p}w_j^2.
$$

Параметр $\alpha \ge 0$ задаёт силу L2-регуляризации. При $\alpha=0$ получается обычная линейная регрессия; при увеличении $\alpha$ веса сильнее уменьшаются, что обычно снижает риск переобучения. Свободный член $w_0$ обычно не включают в сумму штрафа.

Регуляризация особенно полезна, когда признаки имеют разные масштабы или сильно коррелируют друг с другом. Поэтому перед обучением часто применяют стандартизацию: из каждого признака вычитают его среднее и делят на стандартное отклонение. Важно вычислять параметры стандартизации только по train-выборке, чтобы не использовать информацию из validation и test.

#### 4. Итерационное обучение

Вместо готового решателя Ridge можно обновлять веса градиентным спуском. Для функции SSE без учёта деталей реализации направление градиента имеет вид:

$$
\nabla J(\mathbf{w}) = 2X^T(X\mathbf{w} - \mathbf{y}) + 2\alpha\mathbf{w},
$$

где компоненту, соответствующую свободному члену, в регуляризационном слагаемом следует обнулить. На каждой итерации выполняется шаг

$$
\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \eta\nabla J(\mathbf{w}^{(t)}),
$$

где $\eta$ — скорость обучения. Слишком большое значение $\eta$ может вызвать расходимость, а слишком маленькое — очень медленное обучение.

На каждой итерации нужно сохранить SSE, MSE и RMSE отдельно для train и validation. Обучение прекращается, когда выполнено условие $|J_t - J_{t-1}| < \varepsilon$ или достигнуто максимальное число итераций. Поскольку validation-выборка не участвует в обновлении весов, она показывает, как модель обобщает данные.

#### 5. Как определить состояние модели

* **Обучение продолжается успешно:** ошибки train и validation уменьшаются и затем стабилизируются, а разрыв между ними невелик.
* **Переобучение:** ошибка train продолжает уменьшаться, но ошибка validation после некоторого момента растёт. Можно уменьшить сложность модели, усилить Ridge-регуляризацию или остановить обучение раньше.
* **Недообучение:** ошибки на обеих выборках остаются большими и похожими. Можно ослабить регуляризацию, подобрать скорость обучения и число итераций или улучшить признаки.

Тестовая выборка используется только после выбора и обучения модели. Её нельзя применять для подбора параметров, иначе итоговая оценка будет слишком оптимистичной.


#### Шаг 1. Скачайте датасет, откройте его и выведите таблицу на экран

In [1]:
# Ваш код

#### Шаг2. Разделите выборку на тренировочную валидационную и тестовую:
* в тренировочной: 100 первых квартир;
* в валидационной: 50 следующих квартир; 
* в тестовой: 50 последних квартир.
* в каждой выборке матрица X - это столбцы rooms, total area, floor, complex rating, а y - столбец price. 

Названия: X_train, y_train, X_val, y_val, X_test, y_test. 

In [2]:
# Ваш код

#### Шаг 3. Устанавите sklearn чтобы обучать модель Ridge-регрессии

In [ ]:
# Запустите код, чтобы библиотека установилась
!pip install scikit-learn

#### Шаг 4. Обучите модель линейной регрессии на тренировочной выборке с помощью Ridge и sparse_cg с ошибкой, которая вычисляется по формуле SSE.

In [4]:
# Ваш код

#### Шаг 5. Выведите на экран графики ошибок (SSE, MSE, RMSE) в зависимости от итераций для train и val выборок, чтобы отследить их траекторию.

Для этого задания нужно расписать алгоритм обучения вручную, так как функция Ridge в питоне внутри себя эти ошибки не сохраняет.

В самом алгоритме нужно считать и сохранять SSE, MSE, RMSE на каждом шаге итерации для train и для val выборки.

Перед началом алгоритма нужно задать эпсилон - значение ошибки, после которой обучение останавливается, и количество максимальных итераций.

Обучение должно останавливаться тогда, когда выполнен один из двух критериев остановки: либо модуль ошибки меньше эпсилон, либо количество итераций дошло до максимального числа.

* SSE - сумма всех квадратичных отклонений
* MSE - средеее квадратичное отклонение
* RMSE - среднее отклонение предсказания ответа

Подумайте, какие значения для нашего датасета у этих ошибок могут быть допустимыми, чтобы утверждать, что модель обучилась качественно.

После подсчета всех ошибок на всех итерациях нужно вывести на экран три графика:
* график SSE в зависимости от номера итерации: две линии (для train и val)
* график MSE в зависимости от номера итерации: две линии (для train и val)
* график RMSE в зависимости от номера итерации: две линии (для train и val)

In [5]:
# Ваш код

#### Ответьте на вопрос, какую модель Вы получили: обученную, переобученную или недообученную?

* если на графиках все три ошибки для обеих выборок падают - модель обучилась хорошо
* если на графиках ошибки на train падают, а на val с какого-то момента растут - модель переобучилась (подумайте, что стоит изменить в этом случае)
* если на графиках обе ошибки высокие - модель недообучилась (подумайте, что стоит изменить в этом случае)

#### Шаг 6. Применяем модель на тестовой выборке, чтобы предсказать цены квартир.

1. Найдите y_predict

2. Для каждой квартиры из test выведите на экран:
* id квартиры
* её предсказанную цену
* её настоящую цену
* разницу между предсказанной и настоящей ценой (чтобы оценить насколько хорошо Ваша модель предсказывает цены)

3. сохранить получившуюся таблицу

P.S. В итоговой таблице все столбцы должны быть подписаны.

In [6]:
# Ваш код

### Поздравляю! Вы обучили модель и уже сделали первый шаг в машинном обучении! Молодцы!